# BloodMNIST Experiments (CNN or MLP)
Use this notebook as the runner. Shared code lives in the `ml/` package.

In [1]:
from ml.data import build_bloodmnist_dataloaders
from ml.models import CNN, MLP
from ml.setup import get_device, make_criterion, make_optimizer
from ml.training import EarlyStopping, train_with_validation, evaluate_accuracy

In [2]:
BATCH_SIZE = 128
data_bundle = build_bloodmnist_dataloaders(batch_size=BATCH_SIZE, download=True)

train_loader = data_bundle['train_loader']
val_loader = data_bundle['val_loader']
test_loader = data_bundle['test_loader']
n_channels = data_bundle['n_channels']
n_classes = data_bundle['n_classes']

print(f'n_channels={n_channels}, n_classes={n_classes}')

n_channels=3, n_classes=8


In [3]:
MODEL_NAME = 'cnn'   # choose: 'cnn' or 'mlp'
OPTIMIZER_NAME = 'adam'  # choose: 'adam' or 'sgd'
LEARNING_RATE = 0.0003 # 0.01 for sgd
MOMENTUM = 0.9
DROPOUT = 0.5
PATIENCE = 15
EARLY_STOPIING_DELTA = 0.0002
WEIGHT_DECAY = 2e-4

device = get_device()

if MODEL_NAME == 'cnn':
    model = CNN(in_channels=n_channels, num_classes=n_classes, dropout=DROPOUT).to(device)
elif MODEL_NAME == 'mlp':
    model = MLP(input_dim=n_channels * 28 * 28, num_classes=n_classes, dropout=DROPOUT).to(device)
else:
    raise ValueError('MODEL_NAME must be cnn or mlp')

criterion = make_criterion('cross_entropy')
optimizer = make_optimizer(
    model,
    name=OPTIMIZER_NAME,
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
early_stopping = EarlyStopping(PATIENCE, EARLY_STOPIING_DELTA)

print(f'Device: {device}')
print(f'Model: {MODEL_NAME}')
print(f'Optimizer: {OPTIMIZER_NAME}, lr={LEARNING_RATE}')

Device: cpu
Model: cnn
Optimizer: adam, lr=0.0003


In [4]:
history = train_with_validation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=500,
    early_stopping=early_stopping,
)

print('Stopped epoch:', history['stopped_epoch'])
print('Best epoch:', history['best_epoch'])

Epoch 1	Training Loss: 1.1977	Validation Loss: 0.6435
Epoch 2	Training Loss: 0.6052	Validation Loss: 0.4561
Epoch 3	Training Loss: 0.4850	Validation Loss: 0.3860
Epoch 4	Training Loss: 0.4195	Validation Loss: 0.3347
Epoch 5	Training Loss: 0.3549	Validation Loss: 0.3026
Epoch 6	Training Loss: 0.3241	Validation Loss: 0.2757
Epoch 7	Training Loss: 0.2903	Validation Loss: 0.2508
Epoch 8	Training Loss: 0.2696	Validation Loss: 0.2374
Epoch 9	Training Loss: 0.2438	Validation Loss: 0.2290
Epoch 10	Training Loss: 0.2212	Validation Loss: 0.2576
Epoch 11	Training Loss: 0.2180	Validation Loss: 0.2174
Epoch 12	Training Loss: 0.1947	Validation Loss: 0.2196
Epoch 13	Training Loss: 0.1786	Validation Loss: 0.2299
Epoch 14	Training Loss: 0.1736	Validation Loss: 0.2084
Epoch 15	Training Loss: 0.1618	Validation Loss: 0.2061
Epoch 16	Training Loss: 0.1619	Validation Loss: 0.1984
Epoch 17	Training Loss: 0.1491	Validation Loss: 0.2165
Epoch 18	Training Loss: 0.1325	Validation Loss: 0.2071
Epoch 19	Training L

In [5]:
test_accuracy = evaluate_accuracy(model, test_loader, device)
print(f'Test accuracy: {test_accuracy:.2f}%')

Test accuracy: 93.48%
